# Arena GSplat — Colab pipeline

Photos **or** video → COLMAP (one clean solve) → Gaussian Splatting (gsplat) → `.ply`.

**How resuming works.** Every stage writes its outputs and a `state.json` to your
Google Drive. If Colab disconnects, just re-open this notebook, run the setup
cells again, and re-run from where you stopped — finished stages are detected and
skipped automatically. Training also checkpoints to Drive, so an interrupted run
still leaves you a usable `.ply`.

Run the cells top to bottom. After a reconnect: run cells 1–3 (setup), then the
stage cell you need.


## 1 · Setup: Drive, repos, and project name

In [ ]:
# Mount Drive (this is where everything is saved so it survives disconnects)
from google.colab import drive
drive.mount('/content/drive')

# --- EDIT THESE -------------------------------------------------------
GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO_NAME   = "arena-gsplat"
PROJECT     = "my_scene"          # name this capture; becomes a Drive subfolder
DRIVE_ROOT  = "/content/drive/MyDrive/gsplat"   # all projects live under here
# ----------------------------------------------------------------------

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Clone (or update) this pipeline repo
import subprocess, pathlib
repo_dir = f"/content/{REPO_NAME}"
if not pathlib.Path(repo_dir).exists():
    subprocess.run(["git","clone",
                    f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git", repo_dir], check=True)
else:
    subprocess.run(["git","-C",repo_dir,"pull"], check=True)

# Clone gsplat (provides the official simple_trainer.py)
GSPLAT_DIR = "/content/gsplat"
if not pathlib.Path(GSPLAT_DIR).exists():
    subprocess.run(["git","clone","--recursive",
                    "https://github.com/nerfstudio-project/gsplat.git", GSPLAT_DIR], check=True)

import sys
sys.path.insert(0, repo_dir)
print("Setup paths ready.")


## 2 · Install dependencies (COLMAP + gsplat + libs)

In [ ]:
# COLMAP is a system binary; the rest are pip packages.
import subprocess
subprocess.run(["apt-get","-qq","install","-y","colmap","ffmpeg"], check=True)

# Python deps for the pipeline + gsplat examples
subprocess.run(["pip","-q","install","-r",f"/content/{REPO_NAME}/requirements.txt"], check=True)
subprocess.run(["pip","-q","install","-e",GSPLAT_DIR], check=True)
subprocess.run(["pip","-q","install","-r",f"{GSPLAT_DIR}/examples/requirements.txt"], check=True)

import torch
print("CUDA available:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("If CUDA is False: Runtime -> Change runtime type -> GPU, then re-run.")


## 3 · Point at your input

Put your data on GitHub (a `.zip` of images, or a video file) and paste a **raw**
URL below, or upload directly to Colab. The file just needs to land at `INPUT`.


In [ ]:
import urllib.request, pathlib

# Option A: download from a GitHub raw URL (images .zip or a video file)
INPUT_URL = ""   # e.g. "https://github.com/USER/REPO/raw/main/scene.zip" or ".../walk.mp4"

# Option B: leave INPUT_URL empty and upload manually:
#   from google.colab import files; files.upload()   # then set INPUT to the filename

INPUT = "/content/input_data"
if INPUT_URL:
    ext = pathlib.Path(INPUT_URL).suffix or ".zip"
    INPUT = f"/content/input_data{ext}"
    print("Downloading", INPUT_URL)
    urllib.request.urlretrieve(INPUT_URL, INPUT)
    print("Saved ->", INPUT)
else:
    print("No URL set. Upload a file and set INPUT to its path before continuing.")

# Is this a video? None = auto-detect from the extension.
IS_VIDEO = None


## 4 · Build the pipeline object (shows current progress)

In [ ]:
from scripts.pipeline import Pipeline
pipe = Pipeline(project=PROJECT, drive_root=DRIVE_ROOT, gsplat_dir=GSPLAT_DIR)
# The printed summary tells you which stages are already done on Drive.


## 5 · Stage 1 — Ingest (images or video → frames)

For video, `fps` controls how many frames you extract. Set `blur_drop_fraction`
to e.g. `0.15` to discard the blurriest 15% of frames. Re-run with `force=True`
to redo this stage.


In [ ]:
pipe.stage_ingest(
    source=INPUT,
    is_video=IS_VIDEO,
    fps=2.0,                 # video only
    max_images=0,            # 0 = keep all
    blur_drop_fraction=0.0,  # 0.0-0.9
    resize_max_dim=1600,     # long-edge px cap (0 = original)
    # force=True,
)


## 6 · Stage 2 — COLMAP (one clean solve, auto matcher)

Matcher is chosen automatically: **sequential** for video, **exhaustive** for
≤150 images, **vocab-tree** for larger sets. No model merging. If fewer than
~half your images register, the capture has weak overlap — see `docs/troubleshooting.md`.


In [ ]:
pipe.stage_colmap(
    camera_model="OPENCV",   # SIMPLE_RADIAL if you know it's a clean phone lens
    single_camera=True,      # all images from the same camera?
    matcher=None,            # None = auto-select
    use_gpu=True,
    # force=True,
)


## 7 · Stage 3 — Train (gsplat simple_trainer)

30,000 steps is the production default (~15–30 min on a good GPU). Use 7000 for a
quick test. If you hit out-of-memory, set `data_factor=2`. Checkpoints/PLYs are
written to Drive at quarter-step intervals, so a disconnect is recoverable.


In [ ]:
pipe.stage_train(
    max_steps=30000,
    data_factor=1,           # raise to 2 or 4 if VRAM is tight
    strategy="default",      # or "mcmc" for a fixed Gaussian budget
    # force=True,
)


## 8 · Stage 4 — Export the final `.ply` and download

In [ ]:
final = pipe.stage_export()
print("Final PLY:", final)

# If training was interrupted, this still works — it converts the latest
# checkpoint to PLY. Download it:
from google.colab import files
if final:
    files.download(str(final))


## View your splat

- **SuperSplat** (no install): https://superspl.at/editor — drag the `.ply` in.
- **PlayCanvas / Unity** ([UnityGaussianSplatting](https://github.com/aras-p/UnityGaussianSplatting)) for walk-throughs.

The `.ply` and `state.json` live under `DRIVE_ROOT/PROJECT/` on your Drive.
